# 99.7g Forum post 1235: negative concentrations from a square isotherm

## Contents

**Tools:** CADET-Python; ??.

**Python used:** Imports, ??.

**Introduces:** ??.

1. ??
2. ??

Spatial discretization affects the behaviour of an approximation error. In this lesson, we investigate a forum post on a crashing simulation. The root cause are negative values from linear high order approximations of the DG method.

## Forum thread: Negative elution with Langmuir
[Forum post 1235](https://forum.cadet-web.de/t/negative-elution-with-langmuir-mpm/1235) reported the following issue:

### Expected behavior
Fit Langmuir MPM parameters (offline) to match desired pH dependence. Use fit params to simulate bind, wash, and elute.
### Actual behavior
With certain combinations of MPM params (Ka, gamma, beta), no binding occurs and during elution the outlet concentration dives negative. 

### Description of the model
A general rate model with a **strongly favourable Langmuir isotherm** and
**slow pore diffusion**. 

% todo: fill out the relevant model setup, but keep it concise, focus on non standard configs and the aspects that will be relevant later. is it a system, what transport model, what binding model, what elution conditions etc.

### Investigation:
% todo: add tasks for the interactive code below that showcases and visualizes the issue. some configs should show the issue, interaction should show how it behaves when the relevant parameters are changed.
- Lower the feed concentration and watch the negative values appear
- Compare DG and FV at the same number of radial points

In [ ]:
% todo: show where the solution goes below zero, i.e. in the particles. maybe plotting the particle gradient makes sense
import os
import sys
root = os.path.abspath(os.curdir)
while not os.path.isdir(os.path.join(root, 'utility')):
    root, previous = os.path.dirname(root), root
    if root == previous:
        raise RuntimeError('no folder containing utility/ above this notebook')
os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)
    
from pathlib import Path
from IPython.display import display

import utility.cadet_runtime as cadet_runtime
from utility.negative_concentration_explorer import (
    negative_concentration_explorer, shrinking_core_case)

cadet_runtime.configure()

# Every setting is simulated live: a resolved particle costs a few
# seconds per setting, so "Fill this ladder" takes a moment. The
# negative values are read from the particle solution, not from the
# outlet -- which is the point of this case.
shrinking_core_explorer = negative_concentration_explorer(
    shrinking_core_case(),
    output_directory=(Path('generated') / 'user_stories'
                      / 'forum_post_1235'),
)
display(shrinking_core_explorer)

### Fix and explanation

Everything that reaches the solid is adsorbed where it
lands, so the bead fills from the outside in: a saturated outer shell, a sharp
front, and an untouched core. How steep that front is depends on

$$\lambda = \frac{(1 - \varepsilon_p)\, q_{max}}{\varepsilon_p\, c_{feed}}$$

how much more the solid holds than the pore liquid it sits in.
**Lowering the feed raises $\lambda$** and steepens the front inside the bead,
until the radial discretization can no longer represent it.

- the undershoot is in the **pore concentration**, inside the particle
- the **chromatogram at the outlet looks perfectly reasonable** while it happens
- nothing crashes, so **nobody is told**
- the two methods are compared at an **identical number of radial points**


A strongly favourable isotherm is reported in the thread as *negative elution*: concentrations that come out below zero with nothing obviously wrong in the model. The same thing happens inside the particle, where it is easier to provoke and harder to notice -- **nothing crashes, and the outlet looks fine.**